In [1]:
%pip install -U langgraph


Note: you may need to restart the kernel to use updated packages.


In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv
import re
from pydantic import BaseModel

In [5]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

api_key = getpass("Cole sua OpenAI API Key: ").strip()

os.environ["OPENAI_API_KEY"] = api_key

llm = ChatOpenAI(
    model="gpt-5.4-mini",
    api_key=api_key
)

In [7]:
class State(TypedDict, total=False):
    proposal: str
    topic: str
    relevance_score: float
    methodology_score: float
    feasibility_score: float
    impact_score: float
    final_score: float | None
    status: str


class Avaliacao(BaseModel):
    nota: float
    justificativa: str


avaliador = llm.with_structured_output(Avaliacao)


In [4]:
#Funções de Avaliação
#Esta célula define as funções utilizadas no processo de avaliação, incluindo a extração de pontuações e os componentes individuais de avaliação.

In [37]:
def extract_score(content: str) -> float:
    """Extrai a pontuação numérica da resposta do LLM."""
    match = re.search(r'Pontuação:\s*(\d+(?:[.,]\d+)?)', content)

    if match:
        score = float(match.group(1).replace(",", "."))

        if 0 <= score <= 1:
            return score

    raise ValueError(f"Não foi possível extrair uma pontuação válida de: {content}")

def check_relevance(state: State) -> State:
    """Verifica a relevância da proposta científica."""

    criterio = "Aderência da proposta ao tema científico"

    prompt = ChatPromptTemplate.from_template(
        "Analise a relevância da seguinte proposta de pesquisa. "
        "Tema: {topic}. "
        "Critério: {criterio}. "
        "Considere a importância do problema e a clareza dos objetivos. "
        "Forneça uma pontuação de relevância entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nProposta: {proposal}"
    )

    result = llm.invoke(prompt.format(
        topic=state["topic"],
        criterio=criterio,
        proposal=state["proposal"]
    ))

    try:
        state["relevance_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em check_relevance: {e}")
        state["relevance_score"] = 0.0

    return state

def check_methodology(state: State) -> State:
    """Avalia o rigor metodológico da proposta científica."""
    prompt = ChatPromptTemplate.from_template(
        "Analise a metodologia da seguinte proposta de pesquisa. "
        "Considere o rigor científico, a fundamentação teórica, "
        "a adequação dos métodos e a coerência com os objetivos. "
        "Forneça uma pontuação de metodologia entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nProposta: {proposal}"
    )

    result = llm.invoke(
        prompt.format(proposal=state["proposal"])
    )

    try:
        state["methodology_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em check_methodology: {e}")
        state["methodology_score"] = 0.0

    return state
    
def check_feasibility(state: State) -> State:
    """Avalia a viabilidade de execução do projeto científico."""
    prompt = ChatPromptTemplate.from_template(
        "Analise a viabilidade da seguinte proposta de pesquisa. "
        "Considere os recursos necessários, as limitações técnicas, "
        "a complexidade de implementação e a possibilidade de execução. "
        "Forneça uma pontuação de viabilidade entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nProposta: {proposal}"
    )
    result = llm.invoke(prompt.format(
        proposal=state["proposal"]
    ))
    try:
        state["feasibility_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em check_feasibility: {e}")
        state["feasibility_score"] = 0.0
    return state

def evaluate_impact(state: State) -> State:
    """Avalia o impacto científico esperado da proposta."""
    prompt = ChatPromptTemplate.from_template(
        "Avalie o impacto científico da seguinte proposta de pesquisa. "
        "Considere a originalidade, a contribuição para o conhecimento, "
        "o potencial de inovação e a relevância dos resultados esperados. "
        "Forneça uma pontuação de impacto entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nProposta: {proposal}"
    )
    result = llm.invoke(prompt.format(
        proposal=state["proposal"]
    ))
    try:
        state["impact_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em evaluate_impact: {e}")
        state["impact_score"] = 0.0
    return state

def calculate_final_score(state: State) -> State:
    """Calcula a pontuação final da proposta científica."""
    state["final_score"] = (
        state["relevance_score"] * 0.20 +
        state["methodology_score"] * 0.35 +
        state["feasibility_score"] * 0.25 +
        state["impact_score"] * 0.20
    )
    return state

    if any(state.get(c) is None for c in criterios):
        return {
            "final_score": None,
            "status": "Avaliação interrompida"
        }

   
    return {
        "final_score": final_score,
        "status": "Avaliação completa"
    }

In [38]:
print(check_methodology)

<function check_methodology at 0x0000023E9AC24CC0>


In [39]:
#Definição do fluxo de trabalho
#Essa célula define o fluxo de trabalho de avaliação usando StateGraph.

In [40]:
workflow = StateGraph(State)

# Adiciona nós
workflow.add_node("check_relevance", check_relevance)
workflow.add_node("check_methodology", check_methodology)
workflow.add_node("check_feasibility", check_feasibility)
workflow.add_node("evaluate_impact", evaluate_impact)
workflow.add_node("calculate_final_score", calculate_final_score)

# Define arestas condicionais
workflow.add_conditional_edges(
    "check_relevance",
    lambda x: (
        "check_methodology"
        if x["relevance_score"] > 0.5
        else "calculate_final_score"
    )
)

workflow.add_conditional_edges(
    "check_methodology",
    lambda x: (
        "check_feasibility"
        if x["methodology_score"] > 0.6
        else "calculate_final_score"
    )
)

workflow.add_conditional_edges(
    "check_feasibility",
    lambda x: (
        "evaluate_impact"
        if x["feasibility_score"] > 0.7
        else "calculate_final_score"
    )
)
workflow.add_conditional_edges(
    "evaluate_impact",
    lambda x: (
    "calculate_final_score"
    )
)


# Entrada e saída
workflow.set_entry_point("check_relevance")
workflow.add_edge("calculate_final_score", END)

# Compila
app = workflow.compile()

In [41]:
def grade_proposal(proposal: str, topic: str) -> dict:
    """Avalia a proposta científica usando o fluxo de trabalho definido."""
    initial_state = State(
        proposal=proposal,
        topic=topic,
        relevance_score=0.0,
        methodology_score=0.0,
        feasibility_score=0.0,
        impact_score=0.0,
        final_score=None,
        status="Pendente"
    )
    result = app.invoke(initial_state)
    return result

    

In [46]:
sample_proposal = """
O avanço da Inteligência Artificial tem impulsionado o desenvolvimento de modelos de linguagem de grande porte (LLMs) e de pequeno porte (SLMs), capazes de realizar tarefas como geração de textos, análise de documentos, programação e processamento de informações. Entretanto, esses modelos apresentam diferentes exigências computacionais, especialmente em relação ao consumo de memória, processamento, energia e tempo de resposta. Nesse contexto, a busca por maior eficiência computacional tornou-se fundamental para viabilizar aplicações de Inteligência Artificial em ambientes com recursos tecnológicos limitados.

Este projeto de pesquisa tem como objetivo investigar e comparar a eficiência computacional de LLMs e SLMs, identificando estratégias capazes de reduzir custos de execução sem comprometer significativamente a qualidade das respostas. A pesquisa pretende analisar como características como número de parâmetros, arquitetura dos modelos e precisão numérica influenciam o desempenho computacional. Além disso, busca compreender em quais situações modelos menores podem oferecer resultados competitivos em relação a modelos de maior porte.

A metodologia será baseada em experimentos computacionais utilizando modelos de linguagem com diferentes tamanhos e configurações. Serão investigadas técnicas de otimização, como quantização, destilação de conhecimento e gerenciamento eficiente de memória. Os experimentos serão realizados em tarefas de processamento de linguagem natural, considerando métricas como latência de inferência, consumo de memória, velocidade de geração de tokens, consumo energético e qualidade das respostas. Os resultados serão comparados utilizando métodos estatísticos e critérios de avaliação previamente estabelecidos.

A pesquisa também investigará a relação entre eficiência computacional e desempenho dos modelos, buscando identificar configurações que apresentem o melhor equilíbrio entre qualidade, velocidade e utilização de recursos. Para isso, serão analisadas diferentes condições de execução, incluindo variações no tamanho das entradas, na quantidade de solicitações e nas configurações de hardware. Essa abordagem permitirá compreender as limitações e vantagens de cada categoria de modelo.

Como resultados esperados, pretende-se identificar técnicas capazes de melhorar a eficiência computacional dos modelos de linguagem e estabelecer critérios para selecionar soluções adequadas a diferentes aplicações. Espera-se também demonstrar em quais condições os SLMs podem representar alternativas viáveis aos LLMs, especialmente em sistemas com restrições de memória, energia e capacidade de processamento.

Portanto, este projeto busca contribuir para o desenvolvimento de soluções de Inteligência Artificial mais eficientes e acessíveis, oferecendo uma análise experimental das relações entre desempenho, qualidade e custo computacional. A pesquisa poderá auxiliar na implementação de modelos de linguagem em aplicações industriais, acadêmicas e sistemas computacionais com recursos limitados, promovendo a utilização mais racional das tecnologias de Inteligência Artificial.
"""

result = grade_proposal(
    sample_proposal,
    "Eficiência Computacional em Modelos de Linguagem de Grande e Pequeno Porte (LLMs e SLMs)"
)



print("Status:", result["status"])

print(f"Pontuação Final: {final_score:.2f}/10\n")
print(f"Relevância: {relevance_score:.2f}/10")
print(f"Metodologia: {methodology_score:.2f}/10")
print(f"Viabilidade: {feasibility_score:.2f}/10")
print(f"Impacto: {impact_score:.2f}/10")


Status: Pendente
Pontuação Final: 8.38/10

Relevância: 9.50/10
Metodologia: 8.40/10
Viabilidade: 8.40/10
Impacto: 7.20/10
